# gcmprocpy Test Notebook
Test all major features: loading, data parsing, plotting, emissions, and performance.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load Datasets
Test lazy loading with dask and the `ModelDataset` container.

In [ ]:
from gcmprocpy import load_datasets, close_datasets
from gcmprocpy.containers import ModelDataset, PlotData

DATA_DIR = '/glade/work/nikhilr/tiegcm3.0/benchmarks/2.5/seasons/decsol_smin/hist'
FILTER = 'sech'

t0 = time.time()
datasets = load_datasets(DATA_DIR, dataset_filter=FILTER)
t_load = time.time() - t0

print(f'Loaded {len(datasets)} files in {t_load:.3f}s')
for mds in datasets:
    print(f'  {mds.filename} — model: {mds.model}, times: {len(mds._time_values)}')

# Verify ModelDataset
assert isinstance(datasets[0], ModelDataset)
assert datasets[0].model == 'TIE-GCM'
print(f'\nhas_time check: {datasets[0].has_time(datasets[0]._time_values[0])}')

## 2. Data Inspection
Test metadata functions: `time_list`, `var_list`, `level_list`, etc.

In [ ]:
from gcmprocpy import time_list, var_list, level_list, lon_list, lat_list, dim_list, var_info, dim_info

times = time_list(datasets)
print(f'Times: {len(times)} — first: {times[0]}, last: {times[-1]}')

variables = var_list(datasets)
print(f'Variables ({len(variables)}): {variables[:10]}...')

levels = level_list(datasets)
print(f'Levels ({len(levels)}): {levels[:5]}...{levels[-5:]}')

lons = lon_list(datasets)
print(f'Longitudes ({len(lons)}): {lons[0]} to {lons[-1]}')

lats = lat_list(datasets)
print(f'Latitudes ({len(lats)}): {lats[0]} to {lats[-1]}')

dims = dim_list(datasets)
print(f'Dimensions: {dims}')

In [ ]:
# Variable info
info = var_info(datasets, 'TN')
for fname, details in info.items():
    if details:
        print(f'{fname}: units={details["attributes"]["units"]}, dims={details["dimensions"]}')
    break  # just show first

## 3. Data Extraction (arr_* functions)
Test all array extraction functions with `PlotData` returns.

In [ ]:
from gcmprocpy import arr_lat_lon, arr_lev_var, arr_lev_lon, arr_lev_lat, arr_lev_time, arr_lat_time, arr_var
from gcmprocpy import get_mtime, get_time

t_val = times[0]
print(f'Using time: {t_val}')
print(f'mtime: {get_mtime(datasets[0].ds, t_val)}')

In [ ]:
# arr_lat_lon — 2D lat/lon slice at a given level
t0 = time.time()
result = arr_lat_lon(datasets, 'TN', t_val, selected_lev_ilev=5.0, plot_mode=True)
print(f'arr_lat_lon: {time.time()-t0:.3f}s')
print(f'  shape={result.values.shape}, unit={result.variable_unit}, lev={result.selected_lev}')
assert isinstance(result, PlotData)

In [ ]:
# arr_var — full variable at a time (lev × lat × lon)
t0 = time.time()
result = arr_var(datasets, 'TN', t_val, plot_mode=True)
print(f'arr_var: {time.time()-t0:.3f}s')
print(f'  shape={result.values.shape}, levs={result.levs.shape}')

In [ ]:
# arr_lev_var — 1D vertical profile at a lat/lon point
t0 = time.time()
result = arr_lev_var(datasets, 'TN', t_val, selected_lat=0.0, selected_lon=0.0, plot_mode=True)
print(f'arr_lev_var: {time.time()-t0:.3f}s')
print(f'  shape={result.values.shape}, levs={result.levs.shape}')

In [ ]:
# arr_lev_lon — 2D lev/lon slice at a given latitude
t0 = time.time()
result = arr_lev_lon(datasets, 'TN', t_val, selected_lat=0.0, plot_mode=True)
print(f'arr_lev_lon: {time.time()-t0:.3f}s')
print(f'  shape={result.values.shape}, levs={result.levs.shape}, lons={result.lons.shape}')

In [ ]:
# arr_lev_lat — 2D lev/lat slice at a given longitude
t0 = time.time()
result = arr_lev_lat(datasets, 'TN', t_val, selected_lon=0.0, plot_mode=True)
print(f'arr_lev_lat: {time.time()-t0:.3f}s')
print(f'  shape={result.values.shape}, levs={result.levs.shape}, lats={result.lats.shape}')

In [ ]:
# arr_lev_time — 2D lev/time at a lat/lon point (time series)
t0 = time.time()
result = arr_lev_time(datasets, 'TN', selected_lat=0.0, selected_lon=0.0, plot_mode=True)
print(f'arr_lev_time: {time.time()-t0:.3f}s')
print(f'  shape={result.values.shape}, levs={result.levs.shape}, mtimes={len(result.mtime_values)}')

In [ ]:
# arr_lat_time — 2D lat/time at a lon/lev (time series)
t0 = time.time()
result = arr_lat_time(datasets, 'TN', selected_lon=0.0, selected_lev_ilev=5.0, plot_mode=True)
print(f'arr_lat_time: {time.time()-t0:.3f}s')
print(f'  shape={result.values.shape}, lats={result.lats.shape}, mtimes={len(result.mtime_values)}')

In [ ]:
# Test with ilev variable (NE)
t0 = time.time()
result = arr_lat_lon(datasets, 'NE', t_val, selected_lev_ilev=5.0, plot_mode=True)
print(f'arr_lat_lon NE (ilev): {time.time()-t0:.3f}s, shape={result.values.shape}')

## 4. Batch Extraction
Test `batch_arr_lat_lon` which extracts multiple variables in one pass.

In [ ]:
from gcmprocpy import batch_arr_lat_lon

t0 = time.time()
results = batch_arr_lat_lon(datasets, ['TN', 'O1', 'NO'], t_val, selected_lev_ilev=5.0, plot_mode=True)
print(f'batch_arr_lat_lon (3 vars): {time.time()-t0:.3f}s')
for name, r in results.items():
    print(f'  {name}: shape={r.values.shape}, unit={r.variable_unit}')

## 5. Emissions
Test the emission calculation functions.

In [ ]:
from gcmprocpy.data_emissions import mkeno53, arr_mkeno53

# Direct calculation
arr_temp = np.array([500.0, 1000.0, 1500.0])
arr_o = np.array([1e10, 1e11, 1e12])
arr_no = np.array([1e7, 1e8, 1e9])
emission = mkeno53(arr_temp, arr_o, arr_no)
print(f'mkeno53 direct: {emission}')

# From datasets (uses batch_arr_lat_lon internally)
t0 = time.time()
result = arr_mkeno53(datasets, 'NO', t_val, selected_lev_ilev=5.0, plot_mode=True)
print(f'\narr_mkeno53: {time.time()-t0:.3f}s')
print(f'  shape={result.values.shape}, unit={result.variable_unit}, name={result.variable_long_name}')

In [ ]:
# Test plot_mode=False returns raw numpy array
result_raw = arr_mkeno53(datasets, 'NO', t_val, selected_lev_ilev=5.0, plot_mode=False)
print(f'plot_mode=False: type={type(result_raw).__name__}, shape={result_raw.shape}')

## 6. Plotting
Test all 6 plot types.

In [ ]:
from gcmprocpy import plt_lat_lon, plt_lev_var, plt_lev_lon, plt_lev_lat, plt_lev_time, plt_lat_time

In [ ]:
# plt_lat_lon
t0 = time.time()
fig = plt_lat_lon(datasets, 'TN', time=t_val, level=5.0)
print(f'plt_lat_lon: {time.time()-t0:.3f}s')
plt.show()

In [ ]:
# plt_lev_var
t0 = time.time()
fig = plt_lev_var(datasets, 'TN', latitude=0.0, time=t_val, longitude=0.0)
print(f'plt_lev_var: {time.time()-t0:.3f}s')
plt.show()

In [ ]:
# plt_lev_lon
t0 = time.time()
fig = plt_lev_lon(datasets, 'TN', latitude=0.0, time=t_val)
print(f'plt_lev_lon: {time.time()-t0:.3f}s')
plt.show()

In [ ]:
# plt_lev_lat
t0 = time.time()
fig = plt_lev_lat(datasets, 'TN', time=t_val, longitude=0.0)
print(f'plt_lev_lat: {time.time()-t0:.3f}s')
plt.show()

In [ ]:
# plt_lev_time
t0 = time.time()
fig = plt_lev_time(datasets, 'TN', latitude=0.0, longitude=0.0)
print(f'plt_lev_time: {time.time()-t0:.3f}s')
plt.show()

In [ ]:
# plt_lat_time
t0 = time.time()
fig = plt_lat_time(datasets, 'TN', level=5.0, longitude=0.0)
print(f'plt_lat_time: {time.time()-t0:.3f}s')
plt.show()

In [ ]:
# plt_lat_lon — polar projections
t0 = time.time()
fig = plt_lat_lon(datasets, 'TN', time=t_val, level=5.0, projection='north_polar')
print(f'plt_lat_lon north_polar: {time.time()-t0:.3f}s')
plt.show()

t0 = time.time()
fig = plt_lat_lon(datasets, 'TN', time=t_val, level=5.0, projection='south_polar')
print(f'plt_lat_lon south_polar: {time.time()-t0:.3f}s')
plt.show()

t0 = time.time()
fig = plt_lat_lon(datasets, 'TN', time=t_val, level=5.0, projection='polar')
print(f'plt_lat_lon polar (both): {time.time()-t0:.3f}s')
plt.show()

In [ ]:
# plt_lat_lon — mercator with coastlines
t0 = time.time()
fig = plt_lat_lon(datasets, 'TN', time=t_val, level=5.0, coastlines=True)
print(f'plt_lat_lon coastlines: {time.time()-t0:.3f}s')
plt.show()

# plt_lat_lon — polar with coastlines
t0 = time.time()
fig = plt_lat_lon(datasets, 'TN', time=t_val, level=5.0, projection='north_polar', coastlines=True)
print(f'plt_lat_lon north_polar coastlines: {time.time()-t0:.3f}s')
plt.show()

t0 = time.time()
fig = plt_lat_lon(datasets, 'TN', time=t_val, level=5.0, projection='polar', coastlines=True)
print(f'plt_lat_lon polar (both) coastlines: {time.time()-t0:.3f}s')
plt.show()

## 7. Save Output
Test saving a plot to file.

In [ ]:
from gcmprocpy import save_output
import os

fig = plt_lat_lon(datasets, 'TN', time=t_val, level=5.0)
save_output('/tmp', 'test_gcmprocpy', 'png', fig)
assert os.path.exists('/tmp/proc/test_gcmprocpy.png')
print('Plot saved to /tmp/proc/test_gcmprocpy.png')
plt.close(fig)

## 8. Unit Conversion
Test the convert_units function with various unit pairs.

In [ ]:
from gcmprocpy.convert_units import convert_units

passed = 0

# Same unit returns unchanged
data = np.array([1.0, 2.0, 3.0])
result, unit = convert_units(data, 'K', 'K')
assert np.array_equal(result, data) and unit == 'K'; passed += 1

# cm/s -> m/s
result, unit = convert_units(np.array([100.0]), 'cm/s', 'm/s')
assert np.allclose(result, [1.0]) and unit == 'm/s'; passed += 1

# cm -> m
result, unit = convert_units(np.array([100.0, 200.0]), 'cm', 'm')
assert np.allclose(result, [1.0, 2.0]) and unit == 'm'; passed += 1

# km -> m
result, unit = convert_units(np.array([1.0]), 'km', 'm')
assert np.allclose(result, [1000.0]) and unit == 'm'; passed += 1

# K -> C
result, unit = convert_units(np.array([273.15, 373.15]), 'K', 'C')
assert np.allclose(result, [0.0, 100.0]) and unit == 'C'; passed += 1

# K -> F
result, unit = convert_units(np.array([273.15]), 'K', 'F')
assert np.allclose(result, [32.0], atol=0.1) and unit == 'F'; passed += 1

# cm-3 -> m-3
result, unit = convert_units(np.array([1e6]), 'cm-3', 'm-3')
assert np.allclose(result, [1e12]) and unit == 'm-3'; passed += 1

# Unknown conversion returns original
result, unit = convert_units(np.array([1.0, 2.0]), 'unknown_unit', 'm/s')
assert np.array_equal(result, [1.0, 2.0]) and unit == 'unknown_unit'; passed += 1

# Scalar input
result, unit = convert_units(5.0, 'km', 'm')
assert result == 5000.0 and unit == 'm'; passed += 1

# erg/g/s -> J/kg/s
result, unit = convert_units(np.array([1e7]), 'erg/g/s', 'J/kg/s')
assert np.allclose(result, [1.0]) and unit == 'J/kg/s'; passed += 1

print(f'Unit conversion: {passed}/10 tests passed')

## 9. Performance Summary
Benchmark loading and extraction to verify optimizations.

In [ ]:
## 10. Cleanup

## 9. Cleanup

In [ ]:
close_datasets(datasets)
print('All datasets closed. All tests complete.')